In [1]:
import json
import sys
sys.path.insert(0, "..")

import joblib
import numpy as np
import pandas as pd

import config
from src.db_io import escribir_tabla_sqlite, leer_tabla_sqlite
from src.features_modelo import features_modelo_b
from src.monto import (
    crecimiento_anualizado, escenarios_desde_errores, mae_mape,
    split_backtesting_temporal,
)
from src.niveles import asignar_niveles_por_poblacion

(config.OUTPUTS_DIR / "models").mkdir(parents=True, exist_ok=True)

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
panel = leer_tabla_sqlite(config.PLATA_DB, "saldos_mensual_plata")
panel["mes"] = pd.to_datetime(panel["mes"])

# SPEC_V2 §6.3: aplica ÚNICAMENTE a clientes con historial en productos de
# inversión (Invesbot, Inversión Virtual, CDT o Fiducuenta) con saldo > 0 en
# algún momento — la POBLACIÓN se define sobre el total de los 4 productos,
# sin cambios respecto al borrador anterior.
# D5: el RESULTADO se reporta descompuesto en dos componentes:
#   app                = Invesbot + Inversión Virtual (comportamiento tipo App)
#   prod_conservadores = CDT + Fiducuenta (saldos que podrían migrar a la App)
COMPONENTES = {
    "app": ["invesbot", "inversion_virtual"],
    "prod_conservadores": ["cdt", "fiducuenta"],
}
PRODUCTOS_INVERSION = COMPONENTES["app"] + COMPONENTES["prod_conservadores"]

panel_inv = panel[panel["producto"].isin(PRODUCTOS_INVERSION)]

# Panel ANCHO cliente-mes: una columna de saldo por componente + el total,
# para que "total" y "suma de componentes" sean la misma cifra por construcción.
panel_comp = (
    panel_inv.assign(componente=panel_inv["producto"].map(
        {p: c for c, ps in COMPONENTES.items() for p in ps}))
    .groupby(["numero_id", "mes", "componente"], as_index=False)["saldo_mes"].sum()
    .pivot(index=["numero_id", "mes"], columns="componente", values="saldo_mes")
    .fillna(0.0)
    .reset_index()
)
for c in COMPONENTES:
    if c not in panel_comp.columns:
        panel_comp[c] = 0.0
panel_comp["saldo_invertido"] = panel_comp[list(COMPONENTES)].sum(axis=1)

con_historial = set(
    panel_comp.loc[panel_comp["saldo_invertido"] > 0, "numero_id"].unique())
panel_comp = panel_comp[panel_comp["numero_id"].isin(con_historial)]

meses_disponibles = np.sort(panel_comp["mes"].unique())
print(f"clientes con historial de inversión: {len(con_historial):,}")
print(f"meses de historia: {len(meses_disponibles)} "
      f"({meses_disponibles[0].astype('datetime64[D]')} -> {meses_disponibles[-1].astype('datetime64[D]')})")
print(f"filas del panel ancho: {len(panel_comp):,}")

clientes con historial de inversión: 219,542
meses de historia: 13 (2025-06-01 -> 2026-06-01)
filas del panel ancho: 2,545,260


In [2]:
def construir_ventana(panel_saldo):
    """mes_ini/mes_fin/meses por cliente, calculados UNA VEZ sobre el panel
    (total). Los dos componentes reutilizan esta MISMA ventana (D5) para que
    la descomposición sea aditiva: total = app + productos_conservadores."""
    agg = panel_saldo.groupby("numero_id").agg(
        mes_ini=("mes", "min"), mes_fin=("mes", "max"))
    agg["meses"] = ((agg["mes_fin"].dt.year - agg["mes_ini"].dt.year) * 12
                    + (agg["mes_fin"].dt.month - agg["mes_ini"].dt.month))
    return agg.reset_index()


def construir_target_componente(panel_ancho, col_saldo, ventana):
    """Crecimiento anualizado de UN componente, evaluado en la ventana
    compartida (mes_ini/mes_fin del TOTAL). Si el componente aún no existía en
    mes_ini (p.ej. el cliente empezó por CDT y todavía no tenía Invesbot), su
    saldo en ese mes es 0 — consistente con "sin registro = saldo 0"."""
    ini = panel_ancho.merge(ventana[["numero_id", "mes_ini"]], on="numero_id")
    ini = ini.loc[ini["mes"] == ini["mes_ini"]].set_index("numero_id")[col_saldo]
    fin = panel_ancho.merge(ventana[["numero_id", "mes_fin"]], on="numero_id")
    fin = fin.loc[fin["mes"] == fin["mes_fin"]].set_index("numero_id")[col_saldo]

    r = ventana.set_index("numero_id").copy()
    r["saldo_ini"] = ini.reindex(r.index).fillna(0.0)
    r["saldo_fin"] = fin.reindex(r.index).fillna(0.0)
    r["crecimiento_12m"] = crecimiento_anualizado(
        r["saldo_ini"], r["saldo_fin"], r["meses"]).to_numpy()
    return r.reset_index()


# §6.3.4: backtesting temporal — entrenar con los primeros N−3 meses, validar
# contra los últimos 3. El split es sobre el panel ancho (misma columna "mes").
panel_train, panel_valid = split_backtesting_temporal(
    panel_comp, "mes", n_meses_validacion=config.MESES_VALIDACION_BACKTEST)

ventana_train = construir_ventana(panel_train)
ventana_full = construir_ventana(panel_comp)

targets_train = {c: construir_target_componente(panel_train, c, ventana_train)
                 for c in COMPONENTES}
targets_full = {c: construir_target_componente(panel_comp, c, ventana_full)
                for c in COMPONENTES}

print(f"train: {panel_train['mes'].nunique()} meses hasta {panel_train['mes'].max().date()}")
print(f"valid: {panel_valid['mes'].nunique()} meses desde {panel_valid['mes'].min().date()}")
for c in COMPONENTES:
    print(f"\n{c} — crecimiento_12m:")
    print(targets_full[c]["crecimiento_12m"].describe().to_string())

train: 10 meses hasta 2026-03-01
valid: 3 meses desde 2026-04-01

app — crecimiento_12m:
count    2.186940e+05
mean     2.618200e+06
std      3.314673e+07
min     -1.800000e+09
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.500000e+09

prod_conservadores — crecimiento_12m:
count    2.186940e+05
mean     2.285112e+06
std      3.863481e+07
min     -2.296645e+09
25%      0.000000e+00
50%      3.152917e+02
75%      2.664851e+05
max      3.986927e+09


In [3]:
from sklearn.ensemble import HistGradientBoostingRegressor

# §6.3.3: predictoras = saldo actual + tendencia histórica (del PROPIO
# componente) + capacidad financiera.
cols_capacidad = features_modelo_b(df.columns)
base_feats = df[["numero_id"] + cols_capacidad]


def matriz(target_df, columnas_referencia=None):
    X = target_df[["numero_id", "saldo_fin", "meses"]].rename(
        columns={"saldo_fin": "saldo_invertido_actual"})
    X["tendencia_invertida"] = (
        (target_df["saldo_fin"] - target_df["saldo_ini"]) / target_df["meses"].replace(0, np.nan))
    X = X.merge(base_feats, on="numero_id", how="left").reset_index(drop=True)
    ids = X.pop("numero_id")
    X = pd.get_dummies(
        X, columns=[c for c in ["desc_segmento", "grupo_edad", "desc_tipo_de_vivienda"]
                    if c in X.columns], dummy_na=False)
    if columnas_referencia is not None:
        X = X.reindex(columns=columnas_referencia, fill_value=False)
    return ids, X


def entrenar_y_backtest(nombre_componente):
    target_train = targets_train[nombre_componente]
    ids_tr, X_tr = matriz(target_train)
    y_tr = target_train["crecimiento_12m"].reset_index(drop=True)
    ok_tr = y_tr.notna().to_numpy()

    modelo = HistGradientBoostingRegressor(random_state=config.RANDOM_STATE)
    modelo.fit(X_tr[ok_tr], y_tr[ok_tr])

    # Backtest: crecimiento OBSERVADO del componente en los meses de
    # validación, reescalado a 12 meses para ser comparable con el target.
    obs_valid = (panel_valid.sort_values("mes")
                 .groupby("numero_id")[nombre_componente].agg(["first", "last"]))
    n_meses_valid = max(panel_valid["mes"].nunique() - 1, 1)
    real_valid = crecimiento_anualizado(
        obs_valid["first"], obs_valid["last"],
        pd.Series(n_meses_valid, index=obs_valid.index))

    mask_bt = ok_tr & ids_tr.isin(real_valid.index).to_numpy()
    pred_valid = pd.Series(modelo.predict(X_tr[mask_bt]), index=ids_tr[mask_bt].to_numpy())
    real_alineado = real_valid.reindex(pred_valid.index)

    metricas = mae_mape(real_alineado.to_numpy(), pred_valid.to_numpy())
    return modelo, X_tr.columns, pred_valid, real_alineado, metricas


modelos, columnas_ref, preds_valid, reales_valid, metricas_comp = {}, {}, {}, {}, {}
for nombre in COMPONENTES:
    m, cols_ref, pv, rv, met = entrenar_y_backtest(nombre)
    modelos[nombre], columnas_ref[nombre] = m, cols_ref
    preds_valid[nombre], reales_valid[nombre] = pv, rv
    metricas_comp[nombre] = met
    print(f"{nombre} — BACKTEST: MAE={met['mae']:,.0f} | MAPE={met['mape']:.2%} "
          f"(sobre {met['n_mape']:,} clientes)")

# El backtest del TOTAL es la suma exacta de los dos componentes (misma
# ventana, mismos clientes): no se entrena un tercer modelo para el total.
idx_comun = preds_valid["app"].index.intersection(preds_valid["prod_conservadores"].index)
pred_valid_total = (preds_valid["app"].reindex(idx_comun)
                    + preds_valid["prod_conservadores"].reindex(idx_comun))
real_valid_total = (reales_valid["app"].reindex(idx_comun)
                    + reales_valid["prod_conservadores"].reindex(idx_comun))
metricas_total = mae_mape(real_valid_total.to_numpy(), pred_valid_total.to_numpy())
metricas_total["n_meses_historia"] = int(len(meses_disponibles))
metricas_total["n_meses_validacion"] = int(config.MESES_VALIDACION_BACKTEST)
metricas_total["n_clientes_con_historial"] = int(len(con_historial))
metricas_total["componentes"] = metricas_comp
print(f"\nTOTAL (app + productos_conservadores) — BACKTEST: "
      f"MAE={metricas_total['mae']:,.0f} | MAPE={metricas_total['mape']:.2%}")

app — BACKTEST: MAE=13,976,975 | MAPE=1329705.72% (sobre 20,235 clientes)


prod_conservadores — BACKTEST: MAE=18,434,687 | MAPE=730364.92% (sobre 136,600 clientes)

TOTAL (app + productos_conservadores) — BACKTEST: MAE=30,067,086 | MAPE=2255809.51%


In [4]:
preds_full, escenarios_comp = {}, {}
for nombre in COMPONENTES:
    ids_full, X_full = matriz(targets_full[nombre], columnas_referencia=columnas_ref[nombre])
    pred_full = pd.Series(modelos[nombre].predict(X_full), index=ids_full.to_numpy())
    preds_full[nombre] = pred_full

    errores = (reales_valid[nombre] - preds_valid[nombre]).dropna().to_numpy()
    esc = escenarios_desde_errores(pred_full, errores)
    esc.columns = [f"monto_{nombre}_12m_conservador" if c == "conservador"
                   else f"monto_{nombre}_12m_optimista" if c == "optimista"
                   else f"monto_{nombre}_12m_base" for c in esc.columns]
    esc["numero_id"] = pred_full.index
    escenarios_comp[nombre] = esc

# Total = suma de los dos componentes en cada escenario (D5: "el export a
# Power BI debe incluir ambas columnas ADEMÁS del total").
esc_total = escenarios_comp["app"].merge(
    escenarios_comp["prod_conservadores"], on="numero_id", how="outer").fillna(0.0)
esc_total["monto_conservador_12m"] = (
    esc_total["monto_app_12m_conservador"] + esc_total["monto_prod_conservadores_12m_conservador"])
esc_total["monto_base_12m"] = (
    esc_total["monto_app_12m_base"] + esc_total["monto_prod_conservadores_12m_base"])
esc_total["monto_optimista_12m"] = (
    esc_total["monto_app_12m_optimista"] + esc_total["monto_prod_conservadores_12m_optimista"])

for nombre, modelo in modelos.items():
    joblib.dump(modelo, config.OUTPUTS_DIR / "models" / f"monto_12m_{nombre}.pkl")
with open(config.OUTPUTS_DIR / "models" / "metricas_monto.json", "w") as f:
    json.dump(metricas_total, f, indent=2)

print(esc_total[["monto_conservador_12m", "monto_base_12m", "monto_optimista_12m",
                 "monto_app_12m_base", "monto_prod_conservadores_12m_base"]]
      .describe().to_string())
print(
    "\n" + "=" * 78 + "\n"
    "SPEC_V2 §6.3 — LIMITACIÓN A DOCUMENTAR EXPLÍCITAMENTE\n"
    + "=" * 78 + "\n"
    f"Con {len(meses_disponibles)} meses de historia NO es posible validar un horizonte "
    "de 12 meses de forma rigurosa ni capturar estacionalidad anual.\n"
    f"El resultado es una EXTRAPOLACIÓN validada únicamente contra un horizonte de "
    f"{config.MESES_VALIDACION_BACKTEST} meses "
    f"(MAE total={metricas_total['mae']:,.0f}, MAPE total={metricas_total['mape']:.1%}).\n"
    "Reportar SIEMPRE como rango [conservador, optimista], NUNCA como cifra única.\n"
    "\nD5 — DESCOMPOSICIÓN: 'app' (Invesbot + Inversión Virtual) es crecimiento en "
    "comportamiento autogestionado, el más análogo a la nueva App; "
    "'productos_conservadores' (CDT + Fiducuenta) es migración potencial de saldos "
    "existentes bajo el supuesto de que ese saldo PODRÍA trasladarse a la App — no es "
    "un hecho, es un techo de oportunidad. Ambas cifras tienen implicaciones de "
    "negocio distintas y se reportan por separado, nunca solo el total.\n"
    "\nRegularización: forward fill (un saldo persiste hasta el siguiente movimiento). "
    "NO se interpoló linealmente: interpolar inventaría movimientos intermedios."
)

       monto_conservador_12m  monto_base_12m  monto_optimista_12m  monto_app_12m_base  monto_prod_conservadores_12m_base
count           2.195420e+05    2.195420e+05         2.195420e+05        2.195420e+05                       2.195420e+05
mean            5.006090e+06    5.162707e+06         5.038587e+06        2.673532e+06                       2.489175e+06
std             5.020545e+07    5.020545e+07         5.020545e+07        3.188392e+07                       3.882980e+07
min            -9.042372e+08   -9.040805e+08        -9.042047e+08       -7.457242e+08                      -9.041458e+08
25%             0.000000e+00    1.566174e+05         3.249742e+04        6.525323e+04                       9.136416e+04
50%             0.000000e+00    1.566174e+05         3.249742e+04        6.525323e+04                       9.136416e+04
75%             1.523557e+06    1.680174e+06         1.556054e+06        6.525323e+04                       9.136416e+04
max             2.484859e+09    

In [5]:
fact = leer_tabla_sqlite(config.ORO_DB, "fact_cliente_score")
fact = fact.merge(esc_total, on="numero_id", how="left")

fact["tiene_historial_inversion"] = fact["numero_id"].isin(con_historial).astype(int)

# SPEC_V2 §6.3: los clientes sin historial reciben monto NULL. NO imputar cero:
# no es cero, es desconocido. Aplica al total Y a los dos componentes.
sin_hist_inv = fact["tiene_historial_inversion"] == 0
cols_monto = (
    ["monto_conservador_12m", "monto_base_12m", "monto_optimista_12m"]
    + [f"monto_{c}_12m_{esc}" for c in ["app", "prod_conservadores"]
       for esc in ["conservador", "base", "optimista"]]
)
for c in cols_monto:
    fact.loc[sin_hist_inv, c] = np.nan

# §6.2: la población con historial se ordena por valor_esperado = score × monto TOTAL
fact["valor_esperado_12m"] = fact["score"] * fact["monto_base_12m"]
con_hist = fact["poblacion"] == "con_historial"
fact.loc[con_hist, "valor_referencia"] = fact.loc[con_hist, "valor_esperado_12m"]
fact.loc[con_hist, "tipo_valor_referencia"] = "valor_esperado_score_x_monto_12m"
# Los que tienen producto pero no historial de inversión no tienen monto:
# se ordenan por score dentro de su misma población.
sin_monto = con_hist & fact["valor_esperado_12m"].isna()
fact.loc[sin_monto, "valor_referencia"] = fact.loc[sin_monto, "score"]

fact["nivel"] = asignar_niveles_por_poblacion(fact, "valor_referencia", "poblacion")
escribir_tabla_sqlite(fact, config.ORO_DB, "fact_cliente_score")

print(pd.crosstab(fact["poblacion"], fact["nivel"]).to_string())
print(f"\ncon monto estimado: {int(fact['monto_base_12m'].notna().sum()):,}")
print(f"sin monto (NULL, no cero): {int(fact['monto_base_12m'].isna().sum()):,}")

nivel               A       B       C       D
poblacion                                    
con_historial  132368  132367  132368  132367
sin_historial   82689   82688   82688   82688

con monto estimado: 219,542
sin monto (NULL, no cero): 640,681
